# Word Stability by Model and Category

This notebook finds the most stable and least stable words in each category for each model, using exact guess accuracy at the selected final step. Stability here means: for a word at the final step, how often the normalized guess exactly matches the normalized original word.

Change `SELECTED_PIPELINES`, `FINAL_STABILITY_STEP`, and `TOP_N` in the setup cell to adjust the comparison.


In [ ]:
from pathlib import Path
import re

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'datasets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'datasets').exists():
    raise FileNotFoundError('Could not find the project root containing datasets/.')
UNIFIED_CSV = PROJECT_ROOT / 'datasets' / 'unified_semantic_drift_results.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data_analysis' / 'results_analysis' / 'word_stability'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Options: ['pipeline_a'], ['pipeline_b'], or ['pipeline_a', 'pipeline_b']
SELECTED_PIPELINES = ['pipeline_b']

# Final step used for the most/least stable word tables.
FINAL_STABILITY_STEP = 10

# Number of most/least stable words to show per model-category-pipeline group.
TOP_N = 10

MODEL_ORDER = [
    'Llama 3.1 8B Instruct',
    'Llama 3.1 70B Instruct',
    'Gemma 3 4B IT',
    'Gemma 3 12B IT',
    'Gemma 3 27B IT',
]

CATEGORY_ORDER = [
    'High-Freq-Concrete',
    'Low-Freq-Concrete',
    'High-Freq-Abstract',
    'Low-Freq-Abstract',
]

PIPELINE_FROM_PROMPT_MODE = {
    'pipeline_A_guess_then_describe_guessed_word': 'pipeline_a',
    'pipeline_B_paraphrase_description': 'pipeline_b',
}

UNIFIED_CSV.exists(), UNIFIED_CSV


## Load and Normalize

Only the columns needed for word-level stability are loaded, which keeps the notebook lighter than reading the full dataset into memory with every column.

In [ ]:
usecols = ['model_name', 'category', 'original_word', 'step', 'guess', 'prompt_mode']
df = pd.read_csv(UNIFIED_CSV, usecols=usecols, dtype='string')

df['pipeline'] = df['prompt_mode'].map(PIPELINE_FROM_PROMPT_MODE).fillna(df['prompt_mode'])
df['step'] = pd.to_numeric(df['step'], errors='coerce')
df = df[df['pipeline'].isin(SELECTED_PIPELINES)].copy()

df.shape

In [ ]:
def normalize_word(value):
    if pd.isna(value):
        return ''
    value = str(value).strip().lower()
    value = re.sub(r'[^a-zA-Z\s\-]', '', value)
    return value.strip()


def normalize_guess(value):
    if pd.isna(value):
        return ''
    value = str(value).strip().lower()
    value = re.sub(r'^the word is\s+', '', value)
    value = re.sub(r'^answer:\s*', '', value)
    value = re.sub(r'^guess:\s*', '', value)
    value = re.sub(r'[^a-zA-Z\s\-]', '', value)
    return value.strip()


df['word_norm'] = df['original_word'].map(normalize_word)
df['guess_norm'] = df['guess'].map(normalize_guess)
df['exact_correct'] = df['word_norm'].eq(df['guess_norm'])

df[['model_name', 'pipeline', 'category', 'original_word', 'step', 'guess', 'exact_correct']].head()

## Step-10 Word-Level Stability Table

Accuracy is computed per model, pipeline, category, original word, and step. The main stability table below is then filtered to `FINAL_STABILITY_STEP`, so the most/least stable outputs answer: which words are most stable at step 10?


In [ ]:
# Aggregation method:
# 1. Compute exact accuracy for each model + pipeline + category + word + step.
# 2. Filter to FINAL_STABILITY_STEP for the main most/least stable word tables.
word_stability_by_step = (
    df.groupby(['model_name', 'pipeline', 'category', 'original_word', 'step'], observed=True)
    .agg(
        exact_accuracy=('exact_correct', 'mean'),
        correct_count=('exact_correct', 'sum'),
        total_count=('exact_correct', 'size'),
        unique_guesses=('guess_norm', 'nunique'),
    )
    .reset_index()
)

word_stability_by_step['model_name'] = pd.Categorical(word_stability_by_step['model_name'], MODEL_ORDER, ordered=True)
word_stability_by_step['category'] = pd.Categorical(word_stability_by_step['category'], CATEGORY_ORDER, ordered=True)
word_stability_by_step = word_stability_by_step.sort_values(['model_name', 'pipeline', 'category', 'original_word', 'step'])
word_stability_by_step.to_csv(OUTPUT_DIR / 'word_stability_by_step.csv', index=False)

word_stability = word_stability_by_step[word_stability_by_step['step'].eq(FINAL_STABILITY_STEP)].copy()
word_stability = word_stability.sort_values(['model_name', 'pipeline', 'category', 'exact_accuracy', 'original_word'])

word_stability.to_csv(OUTPUT_DIR / 'word_stability_step10_all_words.csv', index=False)
# Backward-compatible filename, now intentionally step-10 only.
word_stability.to_csv(OUTPUT_DIR / 'word_stability_all_words.csv', index=False)
word_stability.head()


## Least and Most Stable Words at Step 10

This returns the `TOP_N` lowest-accuracy and highest-accuracy words for every model, pipeline, and category at `FINAL_STABILITY_STEP`.


In [ ]:
group_cols = ['model_name', 'pipeline', 'category']

least_stable = (
    word_stability
    .sort_values(group_cols + ['exact_accuracy', 'unique_guesses', 'original_word'], ascending=[True, True, True, True, False, True])
    .groupby(group_cols, observed=True)
    .head(TOP_N)
    .assign(stability_rank='least_stable')
)

most_stable = (
    word_stability
    .sort_values(group_cols + ['exact_accuracy', 'unique_guesses', 'original_word'], ascending=[True, True, True, False, True, True])
    .groupby(group_cols, observed=True)
    .head(TOP_N)
    .assign(stability_rank='most_stable')
)

stability_extremes = pd.concat([least_stable, most_stable], ignore_index=True)
stability_extremes = stability_extremes[
    ['stability_rank', 'model_name', 'pipeline', 'category', 'step', 'original_word', 'exact_accuracy', 'correct_count', 'total_count', 'unique_guesses']
]

stability_extremes.to_csv(OUTPUT_DIR / 'word_stability_extremes_by_model_category.csv', index=False)
stability_extremes


## Compact View: Single Lowest and Highest Word at Step 10

This table is useful for paper notes: one least-stable and one most-stable word per model, pipeline, and category at `FINAL_STABILITY_STEP`.


In [ ]:
single_least = (
    word_stability
    .sort_values(group_cols + ['exact_accuracy', 'unique_guesses', 'original_word'], ascending=[True, True, True, True, False, True])
    .groupby(group_cols, observed=True)
    .head(1)
    .rename(columns={
        'original_word': 'least_stable_word',
        'exact_accuracy': 'least_stable_accuracy',
        'correct_count': 'least_stable_correct_count',
        'total_count': 'least_stable_total_count',
        'unique_guesses': 'least_stable_unique_guesses',
    })
)

single_most = (
    word_stability
    .sort_values(group_cols + ['exact_accuracy', 'unique_guesses', 'original_word'], ascending=[True, True, True, False, True, True])
    .groupby(group_cols, observed=True)
    .head(1)
    .rename(columns={
        'original_word': 'most_stable_word',
        'exact_accuracy': 'most_stable_accuracy',
        'correct_count': 'most_stable_correct_count',
        'total_count': 'most_stable_total_count',
        'unique_guesses': 'most_stable_unique_guesses',
    })
)

compact_extremes = single_least[group_cols + [
    'step', 'least_stable_word', 'least_stable_accuracy', 'least_stable_correct_count', 'least_stable_total_count', 'least_stable_unique_guesses'
]].merge(
    single_most[group_cols + [
        'most_stable_word', 'most_stable_accuracy', 'most_stable_correct_count', 'most_stable_total_count', 'most_stable_unique_guesses'
    ]],
    on=group_cols,
    how='outer',
)

compact_extremes.to_csv(OUTPUT_DIR / 'word_stability_single_extremes_by_model_category.csv', index=False)
compact_extremes


## Category-Level Step-10 Extremes Averaged Across Models

This table averages step-10 word accuracy across models first, then reports the least and most stable word for each category and selected pipeline.


In [ ]:
# Average across models for each category + pipeline + word at FINAL_STABILITY_STEP.
# This answers: within a category, which word is most/least stable on average across the selected models?
category_word_step10 = (
    word_stability
    .groupby(['pipeline', 'category', 'original_word'], observed=True)
    .agg(
        mean_step10_accuracy_across_models=('exact_accuracy', 'mean'),
        min_model_step10_accuracy=('exact_accuracy', 'min'),
        max_model_step10_accuracy=('exact_accuracy', 'max'),
        model_n=('model_name', 'nunique'),
        total_correct_count=('correct_count', 'sum'),
        total_count=('total_count', 'sum'),
        mean_unique_guesses=('unique_guesses', 'mean'),
    )
    .reset_index()
)

category_group_cols = ['pipeline', 'category']
category_least = (
    category_word_step10
    .sort_values(category_group_cols + ['mean_step10_accuracy_across_models', 'mean_unique_guesses', 'original_word'], ascending=[True, True, True, False, True])
    .groupby(category_group_cols, observed=True)
    .head(1)
    .rename(columns={
        'original_word': 'least_stable_word',
        'mean_step10_accuracy_across_models': 'least_stable_mean_accuracy',
        'min_model_step10_accuracy': 'least_stable_min_model_accuracy',
        'max_model_step10_accuracy': 'least_stable_max_model_accuracy',
        'total_correct_count': 'least_stable_total_correct_count',
        'total_count': 'least_stable_total_count',
        'mean_unique_guesses': 'least_stable_mean_unique_guesses',
    })
)

category_most = (
    category_word_step10
    .sort_values(category_group_cols + ['mean_step10_accuracy_across_models', 'mean_unique_guesses', 'original_word'], ascending=[True, True, False, True, True])
    .groupby(category_group_cols, observed=True)
    .head(1)
    .rename(columns={
        'original_word': 'most_stable_word',
        'mean_step10_accuracy_across_models': 'most_stable_mean_accuracy',
        'min_model_step10_accuracy': 'most_stable_min_model_accuracy',
        'max_model_step10_accuracy': 'most_stable_max_model_accuracy',
        'total_correct_count': 'most_stable_total_correct_count',
        'total_count': 'most_stable_total_count',
        'mean_unique_guesses': 'most_stable_mean_unique_guesses',
    })
)

category_step10_extremes_across_models = category_least[
    category_group_cols + [
        'least_stable_word',
        'least_stable_mean_accuracy',
        'least_stable_min_model_accuracy',
        'least_stable_max_model_accuracy',
        'least_stable_total_correct_count',
        'least_stable_total_count',
        'least_stable_mean_unique_guesses',
        'model_n',
    ]
].merge(
    category_most[
        category_group_cols + [
            'most_stable_word',
            'most_stable_mean_accuracy',
            'most_stable_min_model_accuracy',
            'most_stable_max_model_accuracy',
            'most_stable_total_correct_count',
            'most_stable_total_count',
            'most_stable_mean_unique_guesses',
        ]
    ],
    on=category_group_cols,
    how='outer',
)

category_word_step10.to_csv(OUTPUT_DIR / 'word_stability_step10_by_category_word_across_models.csv', index=False)
category_step10_extremes_across_models.to_csv(OUTPUT_DIR / 'word_stability_step10_category_extremes_across_models.csv', index=False)
category_step10_extremes_across_models


## Category-Level Step-10 Semantic Similarity Averaged Across Models

This table computes the actual step-10 word-vs-guess semantic similarity for every word, averages each word across models, then selects the top word in each category. Use this for slide values such as ?Stability at Step 10 (Semantic Similarity)?.


In [ ]:
import hashlib
import sqlite3
import numpy as np

SEMANTIC_CACHE = (
    PROJECT_ROOT / 'data_analysis' / 'results_analysis' / 'all_guess_similarity_description_similarity'
    / 'cache' / 'embeddings_sentence_transformers_all_minilm_l6_v2.sqlite'
)
SEMANTIC_CACHE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'


def load_cached_embeddings(texts, cache_path=SEMANTIC_CACHE, model_name=SEMANTIC_CACHE_MODEL):
    texts = list(dict.fromkeys(str(text) for text in texts if isinstance(text, str) and text.strip()))
    embeddings = {}
    if not cache_path.exists():
        return embeddings, texts

    def text_hash(text):
        return hashlib.sha256(text.encode('utf-8')).hexdigest()

    conn = sqlite3.connect(cache_path)
    try:
        for start in range(0, len(texts), 800):
            batch = texts[start:start + 800]
            hashes = [text_hash(text) for text in batch]
            hash_to_text = dict(zip(hashes, batch))
            placeholders = ','.join(['?'] * len(batch))
            rows = conn.execute(
                f'''
                SELECT text_hash, text, dim, vector
                FROM embeddings
                WHERE model_name = ? AND text_hash IN ({placeholders})
                ''',
                [model_name, *hashes],
            ).fetchall()
            for text_hash_value, text, dim, vector in rows:
                if hash_to_text.get(text_hash_value) == text:
                    embeddings[text] = np.frombuffer(vector, dtype=np.float32, count=dim)
    finally:
        conn.close()

    missing = [text for text in texts if text not in embeddings]
    return embeddings, missing


def cosine_cached(text_a, text_b, embeddings):
    vec_a = embeddings.get(text_a)
    vec_b = embeddings.get(text_b)
    if vec_a is None or vec_b is None:
        return np.nan
    return float(np.dot(vec_a, vec_b))


step10_rows = df[df['step'].eq(FINAL_STABILITY_STEP)].copy()
step10_rows = step10_rows[step10_rows['pipeline'].isin(SELECTED_PIPELINES)].copy()
texts = pd.concat([step10_rows['word_norm'], step10_rows['guess_norm']]).dropna().astype(str).unique().tolist()
semantic_embeddings, missing_embedding_texts = load_cached_embeddings(texts)

step10_rows['word_guess_semantic_similarity'] = [
    cosine_cached(word, guess, semantic_embeddings)
    for word, guess in zip(step10_rows['word_norm'], step10_rows['guess_norm'])
]

step10_semantic_by_model_word = (
    step10_rows
    .groupby(['model_name', 'pipeline', 'category', 'original_word'], observed=True)
    .agg(
        step10_exact_accuracy=('exact_correct', 'mean'),
        step10_semantic_similarity=('word_guess_semantic_similarity', 'mean'),
        correct_count=('exact_correct', 'sum'),
        total_count=('exact_correct', 'size'),
        semantic_valid_n=('word_guess_semantic_similarity', 'count'),
        unique_guesses=('guess_norm', 'nunique'),
    )
    .reset_index()
)

step10_semantic_by_category_word = (
    step10_semantic_by_model_word
    .groupby(['pipeline', 'category', 'original_word'], observed=True)
    .agg(
        mean_step10_exact_accuracy_across_models=('step10_exact_accuracy', 'mean'),
        mean_step10_semantic_similarity_across_models=('step10_semantic_similarity', 'mean'),
        min_model_semantic_similarity=('step10_semantic_similarity', 'min'),
        max_model_semantic_similarity=('step10_semantic_similarity', 'max'),
        model_n=('model_name', 'nunique'),
        total_correct_count=('correct_count', 'sum'),
        total_count=('total_count', 'sum'),
        semantic_valid_n=('semantic_valid_n', 'sum'),
        mean_unique_guesses=('unique_guesses', 'mean'),
    )
    .reset_index()
)

category_top_semantic_step10 = (
    step10_semantic_by_category_word
    .sort_values(
        ['pipeline', 'category', 'mean_step10_semantic_similarity_across_models', 'mean_step10_exact_accuracy_across_models', 'original_word'],
        ascending=[True, True, False, False, True],
    )
    .groupby(['pipeline', 'category'], observed=True)
    .head(1)
    .reset_index(drop=True)
)

category_top_exact_step10 = (
    step10_semantic_by_category_word
    .sort_values(
        ['pipeline', 'category', 'mean_step10_exact_accuracy_across_models', 'mean_step10_semantic_similarity_across_models', 'original_word'],
        ascending=[True, True, False, False, True],
    )
    .groupby(['pipeline', 'category'], observed=True)
    .head(1)
    .reset_index(drop=True)
)

category_lowest_semantic_step10 = (
    step10_semantic_by_category_word
    .sort_values(
        ['pipeline', 'category', 'mean_step10_semantic_similarity_across_models', 'mean_step10_exact_accuracy_across_models', 'original_word'],
        ascending=[True, True, True, True, True],
    )
    .groupby(['pipeline', 'category'], observed=True)
    .head(1)
    .reset_index(drop=True)
)

category_lowest_exact_step10 = (
    step10_semantic_by_category_word
    .sort_values(
        ['pipeline', 'category', 'mean_step10_exact_accuracy_across_models', 'mean_step10_semantic_similarity_across_models', 'original_word'],
        ascending=[True, True, True, True, True],
    )
    .groupby(['pipeline', 'category'], observed=True)
    .head(1)
    .reset_index(drop=True)
)

step10_semantic_by_model_word.to_csv(OUTPUT_DIR / 'word_stability_step10_semantic_by_model_word.csv', index=False)
step10_semantic_by_category_word.to_csv(OUTPUT_DIR / 'word_stability_step10_semantic_by_category_word_across_models.csv', index=False)
category_top_semantic_step10.to_csv(OUTPUT_DIR / 'word_stability_step10_category_top_semantic_across_models.csv', index=False)
category_top_exact_step10.to_csv(OUTPUT_DIR / 'word_stability_step10_category_top_accuracy_across_models.csv', index=False)
category_lowest_semantic_step10.to_csv(OUTPUT_DIR / 'word_stability_step10_category_lowest_semantic_across_models.csv', index=False)
category_lowest_exact_step10.to_csv(OUTPUT_DIR / 'word_stability_step10_category_lowest_accuracy_across_models.csv', index=False)
pd.Series(missing_embedding_texts, name='missing_text').to_csv(OUTPUT_DIR / 'word_stability_step10_semantic_missing_embedding_texts.csv', index=False)

category_top_semantic_step10


## Browse One Model and Category

These variables can be changed when a specific slice needs to be inspected in more detail.

In [ ]:
SELECT_MODEL = 'Llama 3.1 70B Instruct'
SELECT_CATEGORY = 'High-Freq-Concrete'
SELECT_PIPELINE = SELECTED_PIPELINES[0]

slice_view = word_stability[
    word_stability['model_name'].astype(str).eq(SELECT_MODEL)
    & word_stability['category'].astype(str).eq(SELECT_CATEGORY)
    & word_stability['pipeline'].eq(SELECT_PIPELINE)
].sort_values(['exact_accuracy', 'original_word'])

slice_view

## Optional: Stability by Step

This can help identify whether a word collapses early or only near the end of the chain.

In [ ]:
# The by-step table is computed above because the step-10 stability tables depend on it.
word_stability_by_step.to_csv(OUTPUT_DIR / 'word_stability_by_step.csv', index=False)
word_stability_by_step.head()


## Single Word Accuracy by Step

This answers questions like: for `Apple`, was the word guessed correctly in every instance at every step? For a single chosen word, accuracy at a step is computed as `correct instances / total instances` for that model and pipeline.

In [ ]:
# Aggregation method in this cell:
# 1. Filter to one chosen category + word.
# 2. For each model, pipeline, and step, count how many instances guessed the original word exactly.
# 3. Divide correct_count by total_count. If exact_accuracy is 1.0, that word was guessed correctly in every instance at that step.

SINGLE_WORD_CATEGORY = 'High-Freq-Concrete'
SINGLE_WORD = 'Apple'
SINGLE_WORD_MODELS = MODEL_ORDER

# Can be a single pipeline string like SELECT_PIPELINE or a list like ['pipeline_a', 'pipeline_b'].
SINGLE_WORD_PIPELINES = globals().get('SELECT_PIPELINE', SELECTED_PIPELINES)

if isinstance(SINGLE_WORD_PIPELINES, str):
    single_word_pipelines = [SINGLE_WORD_PIPELINES]
else:
    single_word_pipelines = list(SINGLE_WORD_PIPELINES)

single_word_rows = df[
    df['category'].astype(str).eq(SINGLE_WORD_CATEGORY)
    & df['original_word'].astype(str).str.casefold().eq(SINGLE_WORD.casefold())
    & df['model_name'].astype(str).isin(SINGLE_WORD_MODELS)
    & df['pipeline'].isin(single_word_pipelines)
].copy()

single_word_accuracy_by_step = (
    single_word_rows
    .groupby(['model_name', 'pipeline', 'category', 'original_word', 'step'], observed=True)
    .agg(
        exact_accuracy=('exact_correct', 'mean'),
        correct_count=('exact_correct', 'sum'),
        total_count=('exact_correct', 'size'),
        unique_guesses=('guess_norm', 'nunique'),
    )
    .reset_index()
    .sort_values(['model_name', 'pipeline', 'step'])
)

single_word_summary = (
    single_word_accuracy_by_step
    .groupby(['model_name', 'pipeline'], observed=True)
    .agg(
        mean_accuracy_across_steps=('exact_accuracy', 'mean'),
        min_step_accuracy=('exact_accuracy', 'min'),
        all_steps_perfect=('exact_accuracy', lambda values: bool((values == 1.0).all())),
        observed_steps=('step', 'nunique'),
    )
    .reset_index()
)

safe_single_word = normalize_word(SINGLE_WORD).replace(' ', '_')
single_word_accuracy_by_step.to_csv(OUTPUT_DIR / f'single_word_accuracy_by_step_{safe_single_word}.csv', index=False)
single_word_summary.to_csv(OUTPUT_DIR / f'single_word_accuracy_summary_{safe_single_word}.csv', index=False)

single_word_accuracy_by_step

In [ ]:
# Compact answer to: was the selected word always correct across all steps?
single_word_summary

## Guess Frequency Ranking for One Word

This shows what a selected word was guessed as most often, split by model, pipeline, and step. The defaults inspect `Apple` in `High-Freq-Concrete`.

In [ ]:
GUESS_RANK_CATEGORY = 'High-Freq-Concrete'
GUESS_RANK_WORD = 'Apple'
# Can be a single pipeline string like 'pipeline_a' or a list like ['pipeline_a', 'pipeline_b'].
GUESS_RANK_PIPELINES = SELECTED_PIPELINES
GUESS_RANK_TOP_N = 20

if isinstance(GUESS_RANK_PIPELINES, str):
    guess_rank_pipelines = [GUESS_RANK_PIPELINES]
else:
    guess_rank_pipelines = list(GUESS_RANK_PIPELINES)

word_guess_rows = df[
    df['category'].astype(str).eq(GUESS_RANK_CATEGORY)
    & df['original_word'].astype(str).str.casefold().eq(GUESS_RANK_WORD.casefold())
    & df['pipeline'].isin(guess_rank_pipelines)
].copy()

# Step-level ranking: for each model + pipeline + step, show what this original word was guessed as.
guess_frequency_by_step = (
    word_guess_rows
    .groupby(['model_name', 'pipeline', 'step', 'guess_norm'], observed=True)
    .size()
    .reset_index(name='count')
)

totals = (
    guess_frequency_by_step
    .groupby(['model_name', 'pipeline', 'step'], observed=True)['count']
    .sum()
    .reset_index(name='total_count')
)

guess_frequency_by_step = guess_frequency_by_step.merge(totals, on=['model_name', 'pipeline', 'step'], how='left')
guess_frequency_by_step['percentage'] = guess_frequency_by_step['count'] / guess_frequency_by_step['total_count']
guess_frequency_by_step['is_correct_guess'] = guess_frequency_by_step['guess_norm'].eq(normalize_word(GUESS_RANK_WORD))

guess_frequency_by_step = guess_frequency_by_step.sort_values(
    ['model_name', 'pipeline', 'step', 'count', 'guess_norm'],
    ascending=[True, True, True, False, True],
)

guess_frequency_by_step_top = (
    guess_frequency_by_step
    .groupby(['model_name', 'pipeline', 'step'], observed=True)
    .head(GUESS_RANK_TOP_N)
    .reset_index(drop=True)
)

# Mistakes-only ranking: same grouping, but excludes the correct guess so drift is easier to inspect.
mistake_frequency_by_step_top = (
    guess_frequency_by_step[~guess_frequency_by_step['is_correct_guess']]
    .groupby(['model_name', 'pipeline', 'step'], observed=True)
    .head(GUESS_RANK_TOP_N)
    .reset_index(drop=True)
)

# All-step companion table: keeps the previous overview behavior, but now includes both pipelines by default.
guess_frequency_all_steps = (
    word_guess_rows
    .groupby(['model_name', 'pipeline', 'guess_norm'], observed=True)
    .size()
    .reset_index(name='count')
)
all_step_totals = (
    guess_frequency_all_steps
    .groupby(['model_name', 'pipeline'], observed=True)['count']
    .sum()
    .reset_index(name='total_count')
)
guess_frequency_all_steps = guess_frequency_all_steps.merge(all_step_totals, on=['model_name', 'pipeline'], how='left')
guess_frequency_all_steps['percentage'] = guess_frequency_all_steps['count'] / guess_frequency_all_steps['total_count']
guess_frequency_all_steps['is_correct_guess'] = guess_frequency_all_steps['guess_norm'].eq(normalize_word(GUESS_RANK_WORD))
guess_frequency_all_steps_top = (
    guess_frequency_all_steps
    .sort_values(['model_name', 'pipeline', 'count', 'guess_norm'], ascending=[True, True, False, True])
    .groupby(['model_name', 'pipeline'], observed=True)
    .head(GUESS_RANK_TOP_N)
    .reset_index(drop=True)
)

safe_word = normalize_word(GUESS_RANK_WORD).replace(' ', '_')
guess_frequency_by_step_top.to_csv(OUTPUT_DIR / f'guess_frequency_ranking_{safe_word}_by_step.csv', index=False)
mistake_frequency_by_step_top.to_csv(OUTPUT_DIR / f'mistake_frequency_ranking_{safe_word}_by_step.csv', index=False)
guess_frequency_all_steps_top.to_csv(OUTPUT_DIR / f'guess_frequency_ranking_{safe_word}_all_steps.csv', index=False)

mistake_frequency_by_step_top


## Human Word-Guess Semantic Similarity

This computes the human-study semantic-similarity line used in the combined plot. It compares each original human-study word with the human guess at each generation, then averages by word first and across words second.


In [ ]:
import hashlib
import sqlite3
import numpy as np

HUMAN_MERGED_COUNTS_DIR = PROJECT_ROOT / 'human_study' / 'merged_with_counts'
HUMAN_EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
DEFAULT_MODEL_EMBEDDING_CACHE = (
    PROJECT_ROOT / 'data_analysis' / 'results_analysis' / 'all_guess_similarity_description_similarity_description_bleu'
    / 'cache' / 'embeddings_sentence_transformers_all_minilm_l6_v2.sqlite'
)
HUMAN_EMBEDDING_CACHE = (
    DEFAULT_MODEL_EMBEDDING_CACHE
    if DEFAULT_MODEL_EMBEDDING_CACHE.exists()
    else OUTPUT_DIR / 'human_word_guess_embeddings_all_minilm_l6_v2.sqlite'
)


def load_human_word_guess_rows():
    rows = []
    for path in sorted(HUMAN_MERGED_COUNTS_DIR.glob('Chain_*_merged_with_counts.csv')):
        chain_match = re.search(r'Chain_(\d+)_', path.name)
        chain_id = int(chain_match.group(1)) if chain_match else path.stem
        chain = pd.read_csv(path)
        # Chain 7 contains duplicated idx rows from a merge issue; keep one row per word/idx.
        chain = chain.drop_duplicates('idx', keep='first')
        for row in chain.itertuples(index=False):
            original = getattr(row, 'Gen_1', '')
            category = getattr(row, 'Category', '')
            idx = getattr(row, 'idx')
            if pd.isna(original) or not str(original).strip():
                continue
            rows.append({
                'chain_id': chain_id,
                'idx': idx,
                'category': category,
                'original_word': original,
                'step': 0,
                'guess': original,
            })
            for step in range(1, 11):
                guess = getattr(row, f'Gen_{step + 1}', '')
                rows.append({
                    'chain_id': chain_id,
                    'idx': idx,
                    'category': category,
                    'original_word': original,
                    'step': step,
                    'guess': guess,
                })
    human_rows = pd.DataFrame(rows)
    human_rows['word_norm'] = human_rows['original_word'].map(normalize_word)
    human_rows['guess_norm'] = human_rows['guess'].map(normalize_guess)
    human_rows = human_rows[human_rows['word_norm'].ne('') & human_rows['guess_norm'].ne('')].copy()
    return human_rows


class SimpleEmbeddingCache:
    def __init__(self, path, model_name):
        self.path = Path(path)
        self.model_name = model_name
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.conn = sqlite3.connect(self.path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS embeddings (
                model_name TEXT NOT NULL,
                text_hash TEXT NOT NULL,
                text TEXT NOT NULL,
                dim INTEGER NOT NULL,
                vector BLOB NOT NULL,
                PRIMARY KEY (model_name, text_hash)
            )
            """)
        self.conn.commit()

    @staticmethod
    def text_hash(text):
        return hashlib.sha256(text.encode('utf-8')).hexdigest()

    def get_many(self, texts):
        unique = list(dict.fromkeys(str(text) for text in texts if isinstance(text, str) and text.strip()))
        found = {}
        if not unique:
            return found
        hash_to_text = {self.text_hash(text): text for text in unique}
        hashes = list(hash_to_text)
        for start in range(0, len(hashes), 900):
            batch = hashes[start:start + 900]
            placeholders = ','.join(['?'] * len(batch))
            rows = self.conn.execute(
                f"""
                SELECT text_hash, text, dim, vector
                FROM embeddings
                WHERE model_name = ? AND text_hash IN ({placeholders})
                """,
                [self.model_name, *batch],
            ).fetchall()
            for text_hash, text, dim, vector in rows:
                if hash_to_text.get(text_hash) == text:
                    found[text] = np.frombuffer(vector, dtype=np.float32, count=dim)
        return found

    def put_many(self, embeddings):
        rows = []
        for text, vector in embeddings.items():
            array = np.asarray(vector, dtype=np.float32)
            rows.append((self.model_name, self.text_hash(text), text, int(array.shape[0]), array.tobytes()))
        self.conn.executemany("""
            INSERT OR REPLACE INTO embeddings
            (model_name, text_hash, text, dim, vector)
            VALUES (?, ?, ?, ?, ?)
            """, rows)
        self.conn.commit()

    def ensure(self, texts, batch_size=128):
        unique = list(dict.fromkeys(str(text) for text in texts if isinstance(text, str) and text.strip()))
        found = self.get_many(unique)
        missing = [text for text in unique if text not in found]
        if missing:
            try:
                from sentence_transformers import SentenceTransformer
            except ImportError as exc:
                raise ImportError(
                    'Human semantic similarity requires sentence-transformers. '
                    'Install it or run this notebook in the same environment used for model semantic-similarity analysis.'
                ) from exc
            encoder = SentenceTransformer(self.model_name)
            encoded = encoder.encode(
                missing,
                batch_size=batch_size,
                show_progress_bar=True,
                normalize_embeddings=True,
            )
            new_embeddings = {text: np.asarray(vector, dtype=np.float32) for text, vector in zip(missing, encoded)}
            self.put_many(new_embeddings)
            found.update(new_embeddings)
        return found

    def close(self):
        self.conn.close()


def cosine_from_embeddings(text_a, text_b, embeddings):
    vec_a = embeddings.get(text_a)
    vec_b = embeddings.get(text_b)
    if vec_a is None or vec_b is None:
        return np.nan
    return float(np.dot(vec_a, vec_b))


human_word_guess_rows = load_human_word_guess_rows()
cache = SimpleEmbeddingCache(HUMAN_EMBEDDING_CACHE, HUMAN_EMBEDDING_MODEL)
try:
    texts = pd.concat([human_word_guess_rows['word_norm'], human_word_guess_rows['guess_norm']]).dropna().astype(str).unique().tolist()
    human_embeddings = cache.ensure(texts)
finally:
    cache.close()

human_word_guess_rows['human_word_guess_semantic_similarity'] = [
    cosine_from_embeddings(word, guess, human_embeddings)
    for word, guess in zip(human_word_guess_rows['word_norm'], human_word_guess_rows['guess_norm'])
]

human_semantic_by_word = (
    human_word_guess_rows
    .groupby(['category', 'idx', 'original_word', 'step'], observed=True)
    .agg(
        word_similarity=('human_word_guess_semantic_similarity', 'mean'),
        chain_n=('chain_id', 'nunique'),
        row_n=('human_word_guess_semantic_similarity', 'size'),
    )
    .reset_index()
)

human_semantic_general = (
    human_semantic_by_word
    .groupby('step', observed=True)
    .agg(
        score=('word_similarity', 'mean'),
        words_n=('idx', 'nunique'),
        rows_n=('row_n', 'sum'),
    )
    .reset_index()
)

human_word_guess_rows.to_csv(OUTPUT_DIR / 'human_word_guess_semantic_similarity_rows.csv', index=False)
human_semantic_by_word.to_csv(OUTPUT_DIR / 'human_word_guess_semantic_similarity_by_word.csv', index=False)
human_semantic_general.to_csv(OUTPUT_DIR / 'human_word_guess_semantic_similarity_general.csv', index=False)
human_semantic_general


## General Pipeline Accuracy, Human Study, and Semantic Similarity

This figure overlays six general lines: Pipeline A exact accuracy, Pipeline B exact accuracy, human-study exact accuracy, Pipeline A semantic similarity, Pipeline B semantic similarity, and human-study semantic similarity. The model semantic lines are read from an overview metrics CSV generated by `run_model_comparison_analysis.py`; `SEMANTIC_OVERVIEW_CSV` can be set manually when the required run is not found automatically.


In [ ]:
import matplotlib.pyplot as plt

HUMAN_STUDY_PERCENTAGES_CSV = PROJECT_ROOT / 'human_study' / 'all_chains_grouped_by_category_percentages.csv'
HUMAN_SEMANTIC_GENERAL_CSV = OUTPUT_DIR / 'human_word_guess_semantic_similarity_general.csv'

# A specific metrics CSV can optionally be assigned when several analysis runs exist.
# Example: PROJECT_ROOT / 'data_analysis' / 'results_analysis' / 'all_guess_similarity' / 'tables' / 'overview_metrics_all_selected_pipelines.csv'
SEMANTIC_OVERVIEW_CSV = None
SEMANTIC_METRIC_COL = 'word_guess_semantic_similarity'


def load_all_pipeline_accuracy():
    usecols = ['model_name', 'category', 'original_word', 'step', 'guess', 'prompt_mode']
    plot_df = pd.read_csv(UNIFIED_CSV, usecols=usecols, dtype='string')
    plot_df['pipeline'] = plot_df['prompt_mode'].map(PIPELINE_FROM_PROMPT_MODE).fillna(plot_df['prompt_mode'])
    plot_df['step'] = pd.to_numeric(plot_df['step'], errors='coerce')
    plot_df = plot_df[plot_df['pipeline'].isin(['pipeline_a', 'pipeline_b'])].copy()
    plot_df['word_norm'] = plot_df['original_word'].map(normalize_word)
    plot_df['guess_norm'] = plot_df['guess'].map(normalize_guess)
    plot_df['exact_correct'] = plot_df['word_norm'].eq(plot_df['guess_norm'])
    # Word-balanced aggregation:
    # 1. Compute each word's exact accuracy at each pipeline + step.
    # 2. Average those word-level accuracies so each word contributes equally.
    word_step_accuracy = (
        plot_df.groupby(['pipeline', 'step', 'original_word'], observed=True)
        .agg(word_accuracy=('exact_correct', 'mean'), n=('exact_correct', 'size'))
        .reset_index()
    )
    return (
        word_step_accuracy.groupby(['pipeline', 'step'], observed=True)
        .agg(score=('word_accuracy', 'mean'), words_n=('original_word', 'nunique'), rows_n=('n', 'sum'))
        .reset_index()
        .assign(metric='Exact Accuracy')
    )


def load_general_human_accuracy():
    human = pd.read_csv(HUMAN_STUDY_PERCENTAGES_CSV)
    rows = [{'step': 0, 'score': 1.0}]
    for step in range(1, 11):
        col = f'Correct_Until_Gen_{step + 1}'
        rows.append({'step': step, 'score': human[col].mean() / 100.0})
    return pd.DataFrame(rows)


def load_general_human_semantic_similarity():
    if 'human_semantic_general' in globals() and not human_semantic_general.empty:
        return human_semantic_general.copy()
    if HUMAN_SEMANTIC_GENERAL_CSV.exists():
        return pd.read_csv(HUMAN_SEMANTIC_GENERAL_CSV)
    print('No human semantic-similarity CSV found. Run the human semantic similarity cell above first.')
    return pd.DataFrame(columns=['step', 'score'])


def find_semantic_overview_csv():
    if SEMANTIC_OVERVIEW_CSV is not None:
        return Path(SEMANTIC_OVERVIEW_CSV)
    candidates = sorted(
        (PROJECT_ROOT / 'data_analysis' / 'results_analysis').glob('**/overview_metrics_all_selected_pipelines.csv'),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    for candidate in candidates:
        columns = pd.read_csv(candidate, nrows=0).columns
        if SEMANTIC_METRIC_COL in columns:
            return candidate
    return None


def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors='coerce')
    weights = pd.to_numeric(weights, errors='coerce').fillna(0)
    valid = values.notna()
    if not valid.any():
        return float('nan')
    if weights.loc[valid].sum() == 0:
        return values.loc[valid].mean()
    return (values.loc[valid] * weights.loc[valid]).sum() / weights.loc[valid].sum()


def load_general_semantic_similarity():
    overview_csv = find_semantic_overview_csv()
    if overview_csv is None:
        print('No semantic overview CSV found. Run run_model_comparison_analysis.py with --metrics guess-sim first, or set SEMANTIC_OVERVIEW_CSV manually.')
        return pd.DataFrame(columns=['pipeline', 'step', 'score'])
    sem = pd.read_csv(overview_csv)
    sem = sem[sem['pipeline'].isin(['pipeline_a', 'pipeline_b'])].copy()
    weight_col = 'word_guess_similarity_n' if 'word_guess_similarity_n' in sem.columns else None
    rows = []
    for (pipeline, step), sub in sem.groupby(['pipeline', 'step'], observed=True):
        if weight_col:
            score = weighted_mean(sub[SEMANTIC_METRIC_COL], sub[weight_col])
        else:
            score = pd.to_numeric(sub[SEMANTIC_METRIC_COL], errors='coerce').mean()
        rows.append({'pipeline': pipeline, 'step': step, 'score': score})
    print(f'Using semantic similarity from: {overview_csv}')
    return pd.DataFrame(rows)


accuracy_general = load_all_pipeline_accuracy()
human_general = load_general_human_accuracy()
semantic_general = load_general_semantic_similarity()
human_semantic_general_plot = load_general_human_semantic_similarity()

fig, ax = plt.subplots(figsize=(8, 6))

style = {
    'pipeline_a': {'label': 'Pipeline A Accuracy', 'color': 'blue'},
    'pipeline_b': {'label': 'Pipeline B Accuracy', 'color': 'green'},
}

for pipeline, info in style.items():
    sub = accuracy_general[accuracy_general['pipeline'].eq(pipeline)].sort_values('step')
    ax.plot(
        sub['step'], sub['score'],
        color=info['color'], linewidth=2.6, marker='o', markersize=4,
        label=info['label'],
    )

ax.plot(
    human_general['step'], human_general['score'],
    color='grey', linewidth=2.6, marker='o', markersize=4,
    label='Human Study Accuracy',
)

for pipeline, color in [('pipeline_a', 'blue'), ('pipeline_b', 'green')]:
    sub = semantic_general[semantic_general['pipeline'].eq(pipeline)].sort_values('step')
    if sub.empty:
        continue
    ax.plot(
        sub['step'], sub['score'],
        color=color, linestyle='--', linewidth=2.4, marker='o', markersize=4,
        label=f"{'Pipeline A' if pipeline == 'pipeline_a' else 'Pipeline B'} Semantic Similarity",
    )

if not human_semantic_general_plot.empty:
    human_semantic_general_plot = human_semantic_general_plot.sort_values('step')
    ax.plot(
        human_semantic_general_plot['step'], human_semantic_general_plot['score'],
        color='black', linestyle='--', linewidth=2.4, marker='o', markersize=4,
        label='Human Study Semantic Similarity',
    )

ax.set_title('General Pipeline Accuracy and Word-Guess Semantic Similarity')
ax.set_xlabel('Step')
ax.set_ylabel('Score')
ax.set_xticks(range(0, 11, 2))
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

figure_path = OUTPUT_DIR / 'general_pipeline_accuracy_human_semantic_similarity.png'
fig.savefig(figure_path, dpi=300)
plt.show()

figure_path
